# 10 — HMM Regime Detection

Demonstrates Gaussian Hidden Markov Model regime detection and comparison with the observable Markov chain:

1. **Fit 3-state return HMM** — bull/bear/sideways via Baum-Welch EM
2. **Compare hard-threshold vs HMM soft probabilities** — Viterbi decoding vs adaptive-threshold classification
3. **Fit 2-state volatility HMM** — orthogonal signal from rolling realised vol
4. **Visualise regime probabilities** — posterior probabilities over time
5. **Forecast n-step ahead** — regime-weighted drift/vol via HMM
6. **Compare trajectories** — HMM-enhanced vs observable Markov
7. **Structural validation** — convergence checks, state ordering

**Prerequisites:**
```bash
cd ../research && pip install -e . && cd ../notebooks
```

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from research.data.prices import fetch_historical_prices
from research.models.hmm import (
    baum_welch,
    fit_volatility_hmm,
    initialize_hmm,
    predict,
    viterbi,
    ASSET_PROFILES,
)
from research.models.markov import (
    classify_regime_series,
    estimate_transition_matrix,
)
from research.models.trajectory import (
    RegimeStats,
    compute_trajectory,
    compute_horizon_drift_vol,
)

## 1. Load Prices

Fetch BTC closes and compute returns.

In [ ]:
TICKER = 'BTC'
DAYS = 180
HORIZON = 7

prices = fetch_historical_prices(TICKER, days=DAYS)
prices['returns'] = prices['close'].pct_change()
returns = prices['returns'].dropna().values
current_price = float(prices['close'].iloc[-1])

print(f"Current price: ${current_price:,.2f}")
print(f"Returns: {len(returns)} days")

## 2. Fit 3-State Return HMM

Fit a Gaussian HMM on daily returns. The states are ordered by mean return: state 0 = bearish, state 2 = bullish.

In [ ]:
# Fit with moderate iterations for notebook speed
hmm_result = baum_welch(returns, n_states=3, max_iterations=50, tolerance=1e-3)

print(f"Converged: {hmm_result.converged}")
print(f"Iterations: {hmm_result.iterations}")
print(f"Log-likelihood: {hmm_result.log_likelihood:.2f}")
print()
for i, (mu, sigma) in enumerate(zip(hmm_result.params.means, hmm_result.params.stds)):
    label = {0: 'bear', 1: 'sideways', 2: 'bull'}.get(i, f'state_{i}')
    print(f"State {i} ({label:8s}): mean={mu:+.5f}  std={sigma:.5f}")

print("\nTransition matrix:")
print(pd.DataFrame(hmm_result.params.A, columns=['bear', 'sideways', 'bull'], index=['bear', 'sideways', 'bull']).round(3))

## 3. Compare Hard-Threshold vs HMM Regimes

Observable Markov uses adaptive thresholds; HMM uses soft probabilities. Compare the two approaches.

In [ ]:
# Observable Markov regimes
obs_regimes = classify_regime_series(returns)

# HMM regimes via Viterbi
hmm_states = viterbi(returns, hmm_result.params)
hmm_regimes = []
for s in hmm_states:
    if s == 0:
        hmm_regimes.append('bear')
    elif s == 2:
        hmm_regimes.append('bull')
    else:
        hmm_regimes.append('sideways')

# Agreement rate
agreement = sum(o == h for o, h in zip(obs_regimes, hmm_regimes)) / len(obs_regimes)
print(f"Agreement rate: {agreement*100:.1f}%")

# Regime duration comparison
obs_counts = pd.Series(obs_regimes).value_counts().sort_index()
hmm_counts = pd.Series(hmm_regimes).value_counts().sort_index()
comparison = pd.DataFrame({'observable': obs_counts, 'hmm': hmm_counts}).fillna(0).astype(int)
print(comparison)

# NOTE: Low agreement is expected — the HMM clusters by volatility,
# not return direction. On volatile assets like BTC, it typically finds
# one low-vol "normal" state and a few extreme outlier states.
# This is universal across asset classes (crypto, equity, ETF) because
# Gaussian HMMs maximize likelihood, and one broad Gaussian explains
# the bulk of any return distribution more efficiently than splitting
# by mean. The value is in continuous drift/vol forecasts, not labels.
print("\nNote: HMM clusters by volatility universally, not direction.")
print("See hmm.py docstring and Section 8 for full explanation.")


## 4. Regime Probabilities Over Time

Visualise the posterior regime probabilities from the HMM.

In [ ]:
# Posterior probabilities
from hmmlearn import hmm as hmm_module

model = hmm_module.GaussianHMM(n_components=3, covariance_type='diag', n_iter=1, init_params='')
model.startprob_ = hmm_result.params.pi.copy()
model.transmat_ = hmm_result.params.A.copy()
model.means_ = hmm_result.params.means.reshape(-1, 1).copy()
model.covars_ = (hmm_result.params.stds ** 2).reshape(-1, 1).copy()

probs = model.predict_proba(returns.reshape(-1, 1))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

# Price with regime shading
ax = axes[0]
ax.plot(prices['close'].values, color='black', linewidth=1)
ax.set_ylabel('Price')
ax.set_title(f'{TICKER}: Price with HMM Regimes')
ax.grid(True, alpha=0.3)

# Regime probabilities
ax = axes[1]
ax.stackplot(range(len(probs)), probs[:, 0], probs[:, 1], probs[:, 2],
             labels=['Bear', 'Sideways', 'Bull'],
             colors=['#d62728', '#ffbb78', '#2ca02c'], alpha=0.8)
ax.set_xlabel('Day')
ax.set_ylabel('Probability')
ax.set_title('HMM Posterior Regime Probabilities')
ax.legend(loc='upper left')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Fit 2-State Volatility HMM

Rolling realised volatility as an orthogonal signal.

In [ ]:
vol_scale = fit_volatility_hmm(returns, vol_window=5, n_states=2)
print(f"Volatility scale factor: {vol_scale:.2f}")
print(f"Interpretation: {'elevated' if vol_scale > 1 else 'normal'} vol regime")

## 6. Forecast n-Step Ahead

Use the HMM to forecast regime probabilities and expected return/volatility at horizon.

In [ ]:
hmm_pred = predict(returns, hmm_result.params, forecast_horizon=HORIZON)

print(f"Current state: {hmm_pred.current_state}")
print(f"State probabilities: {hmm_pred.state_probabilities.round(3)}")
print(f"Forecast probabilities (day {HORIZON}): {hmm_pred.forecast_probabilities.round(3)}")
print(f"Expected return (daily): {hmm_pred.expected_return:.5f}")
print(f"Expected volatility (daily): {hmm_pred.expected_volatility:.5f}")

# HMM override for trajectory
profile = ASSET_PROFILES['crypto']
hmm_weight = profile.hmm_weight_multiplier * 0.5  # base weight 0.5, scaled by profile
hmm_override = {
    'drift': hmm_pred.expected_return,
    'vol': hmm_pred.expected_volatility,
    'weight': np.clip(hmm_weight, 0.0, 1.0),
}
print(f"\nHMM override: drift={hmm_override['drift']:.5f}, vol={hmm_override['vol']:.5f}, weight={hmm_override['weight']:.2f}")

## 7. Compare HMM vs Observable Markov Trajectories

Generate trajectories with and without HMM enhancement.

In [ ]:
# Observable Markov transition matrix and stats
obs_regimes = classify_regime_series(returns)
P = estimate_transition_matrix(obs_regimes, decay_rate=profile.decay_rate)
current_regime = obs_regimes[-1]

regime_stats = {}
for state in ['bull', 'bear', 'sideways']:
    mask = np.array([r == state for r in obs_regimes])
    state_returns = returns[mask]
    regime_stats[state] = RegimeStats(
        mean_return=float(np.mean(state_returns)) if len(state_returns) > 0 else 0.0,
        std_return=float(np.std(state_returns, ddof=1)) if len(state_returns) > 1 else 0.01,
    )

# Trajectory without HMM
traj_base = compute_trajectory(current_price, HORIZON, P, regime_stats, current_regime, n_samples=1000, nu=profile.student_t_nu)

# Trajectory with HMM override
traj_hmm = compute_trajectory(current_price, HORIZON, P, regime_stats, current_regime, hmm_override=hmm_override, n_samples=1000, nu=profile.student_t_nu)

# DataFrames
df_base = pd.DataFrame([
    {'day': pt.day, 'expected': pt.expected_price, 'lower': pt.lower_bound, 'upper': pt.upper_bound, 'p_up': pt.p_up}
    for pt in traj_base
])
df_hmm = pd.DataFrame([
    {'day': pt.day, 'expected': pt.expected_price, 'lower': pt.lower_bound, 'upper': pt.upper_bound, 'p_up': pt.p_up}
    for pt in traj_hmm
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price trajectories
ax = axes[0]
ax.fill_between(df_base['day'], df_base['lower'], df_base['upper'], alpha=0.15, label='Base 90% CI')
ax.plot(df_base['day'], df_base['expected'], linewidth=2, label='Base Expected', color='C0')
ax.plot(df_hmm['day'], df_hmm['expected'], linewidth=2, label='HMM Expected', color='C1', linestyle='--')
ax.axhline(current_price, color='red', linestyle=':', alpha=0.5, label=f'Current: ${current_price:,.0f}')
ax.set_xlabel('Day')
ax.set_ylabel('Price')
ax.set_title(f'{TICKER}: Trajectory Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# P(up)
ax = axes[1]
ax.bar(df_base['day'] - 0.2, df_base['p_up'] * 100, width=0.35, label='Base', alpha=0.7, color='C0')
ax.bar(df_hmm['day'] + 0.2, df_hmm['p_up'] * 100, width=0.35, label='HMM', alpha=0.7, color='C1')
ax.axhline(50, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Day')
ax.set_ylabel('P(up) %')
ax.set_title(f'{TICKER}: P(up) Comparison')
ax.set_ylim(0, 100)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. Structural Validation & Understanding HMM Behavior

Replicate the TypeScript integration-test assertions, plus understand
why HMMs on raw returns behave the way they do.

In [ ]:
# Convergence
assert hmm_result.converged, 'HMM did not converge'

# State ordering: means sorted ascending
means = hmm_result.params.means
assert means[0] <= means[1] <= means[2], f'Means not sorted: {means}'

# Transition matrix is stochastic
A = hmm_result.params.A
for row in A:
    assert abs(sum(row) - 1.0) < 1e-5, 'Transition matrix row does not sum to 1'

# Predictions are valid probabilities
assert 0 <= hmm_pred.current_state < 3
assert abs(sum(hmm_pred.state_probabilities) - 1.0) < 1e-5
assert abs(sum(hmm_pred.forecast_probabilities) - 1.0) < 1e-5
for p in hmm_pred.state_probabilities:
    assert 0 <= p <= 1

# Expected values are finite
assert np.isfinite(hmm_pred.expected_return)
assert np.isfinite(hmm_pred.expected_volatility)
assert hmm_pred.expected_volatility >= 0

# Trajectory values are finite and valid
for pt in traj_hmm:
    assert np.isfinite(pt.expected_price)
    assert np.isfinite(pt.lower_bound)
    assert np.isfinite(pt.upper_bound)
    assert pt.lower_bound < pt.expected_price < pt.upper_bound
    assert 0 <= pt.p_up <= 1

# Volatility scale in range
assert 0.5 <= vol_scale <= 2.0

# ---- WHY HMMs ON RAW RETURNS CLUSTER BY VOLATILITY ----
#
# Gaussian HMMs maximize likelihood by fitting Gaussian distributions.
# In financial returns, variance differences between regimes are orders
# of magnitude larger than mean differences. As a result, EM finds that
# one broad "normal" Gaussian explains the bulk of data more efficiently
# than splitting by mean. The outlier days become tiny, high-vol states.
#
# This is UNIVERSAL across asset classes:
#   - Crypto (BTC): ~98% in one state
#   - Equity (AAPL): ~99% in one state
#   - ETF (SPY):    ~100% in one state
#
# The value is NOT in hard regime labels (bull/bear/sideways).
# The value IS in the continuous forecasts:
#   - expected_return (weighted avg of state means)
#   - expected_volatility (weighted avg of state stds)
#
# For explicit regime labels, use the observable Markov model.
# For a cleaner HMM split, use fit_2state_return_hmm (calm/volatile).

print('All structural validations passed.')
print()
print('HMM behavior: This model clusters by VOLATILITY, not direction.')
print('One state dominates (~98-100% of days) because Gaussian EM')
print('maximizes likelihood by fitting a single broad Gaussian to the')
print('bulk of the return distribution. The outlier days are absorbed')
print('as tails. This is universal across asset classes (crypto, equity, ETF).')
print()
print('The value is in continuous expected_return/expected_volatility')
print('forecasts, not hard regime labels.')
